# Description
#### Test of the cell creation based on OSM data. The goal of this excercise is to give engineering a sample of cells to test the query times from the DB.

# Imports

In [1]:
%load_ext autoreload
%autoreload 2
%reset -f
%matplotlib widget

#### These are libraries that I have created to query from the DB faster, they should be replaced by query_mssql as explained after on the code


In [2]:
from locallib.picarrodb import *
from locallib.query import *

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


In [3]:
#Py file where the NOP class is defined
from NOP import *

import matplotlib.pyplot as plt
from shapely import wkt


/home/sandbox/personal-repos/NumberOfPasses/NOP.py:17: UserWarning: registration of accessor <class 'NOP.NOPAccessor'> under name 'nop' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  @pd.api.extensions.register_dataframe_accessor("nop")


# Query the Breadcrumbs and the survey area
Segments: For the current notebook, the breadcrumbs are only used for visualization purposes. 

For this excercise, the SurveyArea is going to be the total area covered by the surveys in the report. (Check the first lines of the next cell). The SurveyArea is needed to extract the OSM roads. In the future, they should be replaced by the Pre-driving assesment.

The segments were obtained using the some local libraries I use for faster extraction. They need to be changed into query_mssql. The query I useed to obtained the segments dataframe is the following

```sql
SELECT 
    RDS.ReportId AS ReportId,
    S.SurveyId,
    S.Shape.STAsText() AS Breadcrumb,
    S.[Order] as [Order]
FROM Segment S
JOIN ReportDrivingSurvey RDS ON S.SurveyId = RDS.SurveyId
WHERE RDS.ReportId IN (SELECT ReportId FROM #TempReport)
```

In [ ]:
#Query the segments for any random report of Cadent. Replace this part with query_mssql to get the segments for all reports.
a = get_reports('Cadent',years = [2026]).execute([EU2_Conn])
report_bc = a.iloc[[700]].copy()
report_bc.db.set_query(query_Segments_byReport(report_table = '#TempReport'))
segments = report_bc.db.execute([EU2_Conn], source_col = 'ReportId', temp_table_name = '#TempReport')

# Create a GeoDataFrame for the segments with geometry from the Breadcrumb column
segments['geometry'] = segments['Breadcrumb'].apply(wkt.loads)
segments_gdf = gpd.GeoDataFrame(segments, geometry='geometry', crs='EPSG:4326')
utm_crs = segments_gdf.estimate_utm_crs()
segments_gdf = segments_gdf.to_crs(utm_crs)
segments_gdf.set_geometry('geometry', inplace=True)
segments_gdf.head()

,ReportId,SurveyId,Breadcrumb,Order,geometry
0,2C524C2E-2EB2-C1EC-338D-3A209D084883,61BBDBD3-24C6-8A47-B6F4-3A2095ACDB2D,"LINESTRING (-0.61622317 53.56803675, -0.616737...",290,"LINESTRING (657855.25 5938105.61, 657818.178 5..."
1,2C524C2E-2EB2-C1EC-338D-3A209D084883,61BBDBD3-24C6-8A47-B6F4-3A2095ACDB2D,"LINESTRING (-0.64204592 53.58533182, -0.641987...",170,"LINESTRING (656081.799 5939972.298, 656087.175..."
2,2C524C2E-2EB2-C1EC-338D-3A209D084883,61BBDBD3-24C6-8A47-B6F4-3A2095ACDB2D,"LINESTRING (-0.64798907 53.58427384, -0.647901...",242,"LINESTRING (655692.355 5939841.607, 655699.551..."
3,2C524C2E-2EB2-C1EC-338D-3A209D084883,61BBDBD3-24C6-8A47-B6F4-3A2095ACDB2D,"LINESTRING (-0.64148749 53.58464635, -0.641497...",235,"LINESTRING (656121.285 5939897.282, 656120.502..."
4,2C524C2E-2EB2-C1EC-338D-3A209D084883,61BBDBD3-24C6-8A47-B6F4-3A2095ACDB2D,"LINESTRING (-0.64090242 53.57766611, -0.643926...",80,"LINESTRING (656185.745 5939122.189, 655986.294..."


: 

# Extract all the streets needed (Pre driving)
### Query the OSM data for cell creation.

In [ ]:
import osmnx as ox
from shapely.geometry import box
from shapely.ops import unary_union

# Build a tight polygon around the actual survey paths (50 m buffer) rather than
# using the raw bounding box, which can be orders of magnitude larger and causes
# osmnx to request too much data, crashing the kernel with an OOM error.
segments_union = segments_gdf.to_crs("EPSG:4326").unary_union
survey_poly = segments_union.buffer(0.0005)   # ~50 m in degrees at mid-latitudes

# Download only the roads that intersect the buffered survey polygon
G = ox.graph_from_polygon(survey_poly, network_type="drive", retain_all=False)

# Convert to GeoDataFrame
nodes, edges = ox.graph_to_gdfs(G)
edges = edges.set_crs("EPSG:4326", allow_override=True)

# Crop each road geometry to the survey polygon (drop anything outside)
edges['geometry'] = edges['geometry'].intersection(survey_poly)
edges = edges[~edges['geometry'].is_empty]
utm_crs = edges.estimate_utm_crs()
edges = edges.to_crs(utm_crs)
edges.set_geometry('geometry', inplace=True)
edges.plot()

# Calculate the grid

In [ ]:
#Prepares the gdf for grid creation. It translated to a (0,0) reference and create the Virtual Road
edges.nop.prepare_survey_gdf()

# Generate the grid based on the Virtual Road
road_grid = edges.nop.generate_grid()

# Create the network where the NOP is calculated, since we are creating the grid using OSM roads, this for the moment is not useful
net = edges.nop.create_network()

# Plot the grid, the segments and the OSM Roads

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))

# Plot the road grid
road_grid.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.5, alpha=0.7, label='Road Grid')

# Plot the segments on top
segments_
segments_gdf.plot(ax=ax, color='red', linewidth=2, alpha=0.7, label='Segments')
edges.plot(ax=ax, color='blue', linewidth=2, alpha=0.7, label='Edges')
ax.legend()
plt.title('Survey Line Geometries with Road Grid Overlap')
plt.xlabel('Easting')
plt.ylabel('Northing')
plt.show()